# Fake Job Detection - National Honor Edition

This is the code that achieved 0.915 on Kaggle!

Focus: **company_profile** extreme feature engineering

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')

print('=' * 80)
print('FAKE JOB - NATIONAL HONOR EDITION')
print('=' * 80)

In [ ]:
train_df = pd.read_csv('job_postings_train.csv')
test_df = pd.read_csv('job_postings_test.csv')

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Fraud ratio: {train_df["fraudulent"].mean():.4f}')

In [ ]:
def extract_features(df):
    df = df.copy()
    
    # Text statistics
    text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
    for col in text_cols:
        df[f'{col}_len'] = df[col].fillna('').apply(len)
        df[f'{col}_wc'] = df[col].fillna('').apply(lambda x: len(str(x).split()))
        df[f'{col}_unique_words'] = df[col].fillna('').apply(lambda x: len(set(str(x).split())))
        df[f'{col}_isna'] = df[col].isna().astype(int)
        df[f'{col}_avglen'] = df[col].fillna('').apply(lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split())>0 else 0)
        df[f'{col}_unique_ratio'] = df[f'{col}_unique_words'] / (df[f'{col}_wc'] + 1)
    
    # --- company_profile features!
    cp_text = df['company_profile'].fillna('')
    df['cp_is_empty'] = (cp_text == '').astype(int)
    df['cp_has_company'] = cp_text.str.lower().str.contains('company').astype(int)
    df['cp_has_inc'] = cp_text.str.lower().str.contains('inc|llc|corp|limited').astype(int)
    df['cp_has_weird'] = (cp_text.fillna('').str.len() < 10).astype(int)
    df['cp_has_company_name'] = cp_text.str.lower().str.contains('is|are|we|our').astype(int)
    df['cp_has_year'] = cp_text.str.contains(r'\b\d{4}\b').astype(int)
    df['cp_has_location'] = cp_text.str.lower().str.contains('worldwide|global|international').astype(int)
    df['cp_num_words'] = cp_text.fillna('').apply(lambda x: len(str(x).split()))
    df['cp_num_chars'] = cp_text.fillna('').apply(len)
    df['cp_num_upper'] = cp_text.fillna('').apply(lambda x: sum(1 for c in x if c.isupper()))
    df['cp_upper_ratio'] = df['cp_num_upper'] / (df['cp_num_chars'] + 1)
    df['cp_has_suspicious_words'] = cp_text.str.lower().str.contains('make money|earn|free|instant|quick').astype(int)
    df['cp_has_websites'] = cp_text.str.contains(r'http|www|\\.com|\\.net|\\.org').astype(int)
    
    # --- Key ratio features
    df['cp_ratio'] = df['company_profile_len'] / (df['description_len'] + 1)
    df['desc_cp_len_diff'] = df['description_len'] - df['company_profile_len']
    df['cp_title_len_sum'] = df['company_profile_len'] + df['title_len']
    df['cp_req_len_sum'] = df['company_profile_len'] + df['requirements_len']
    
    # Other features
    desc = df['description'].fillna('').str.lower()
    df['desc_has_url'] = desc.str.contains(r'http|www\.|url').astype(int)
    df['desc_has_email'] = desc.str.contains(r'@|email').astype(int)
    df['desc_has_phone'] = desc.str.contains(r'phone|call|contact|tel:').astype(int)
    df['desc_has_linkedin'] = desc.str.contains(r'linkedin').astype(int)
    df['desc_has_websites'] = desc.str.contains(r'http|www|\\.com|\\.net').astype(int)
    df['desc_has_free'] = desc.str.contains(r'free|no cost|no fee').astype(int)
    df['desc_has_money'] = desc.str.contains(r'money|cash|pay|fee|salary|income').astype(int)
    df['desc_has_urgent'] = desc.str.contains(r'urgent|immediate|hiring now|asap').astype(int)
    df['desc_excl'] = df['description'].fillna('').apply(lambda x: x.count('!'))
    df['desc_upper_ratio'] = df['description'].fillna('').apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
    df['desc_digit_ratio'] = df['description'].fillna('').apply(lambda x: sum(1 for c in x if c.isdigit()) / (len(x) + 1))
    
    # Salary
    if 'salary_range' in df.columns:
        df['salary_isna'] = df['salary_range'].isna().astype(int)
        def parse_sal(s):
            if pd.isna(s): return (np.nan, np.nan, np.nan)
            try:
                nums = [float(x.replace(',','')) for x in s.replace('-', ' ').split() if x.replace(',','').replace('.','').isdigit()]
                if len(nums)>=2: return min(nums), max(nums), max(nums)-min(nums)
                elif len(nums)==1: return nums[0], nums[0], 0
            except: pass
            return np.nan, np.nan, np.nan
        sal_feats = df['salary_range'].apply(parse_sal)
        df['salary_min'] = [x[0] for x in sal_feats]
        df['salary_max'] = [x[1] for x in sal_feats]
        df['salary_range_val'] = [x[2] for x in sal_feats]
    
    # Location
    if 'location' in df.columns:
        loc = df['location'].fillna('')
        df['loc_len'] = loc.apply(len)
        df['country'] = loc.apply(lambda x: str(x).split(',')[0].strip() if ',' in str(x) else 'UNK')
        df['loc_nparts'] = loc.apply(lambda x: len(str(x).split(',')))
    
    # Category features
    cat_cols = ['employment_type', 'required_experience', 'required_education', 'industry', 'function', 'department', 'country']
    for col in cat_cols:
        if col in df.columns:
            df[f'{col}_isna'] = df[col].isna().astype(int)
            df[col] = df[col].fillna('UNK')
    
    return df

In [ ]:
print('Extracting features...')
train = extract_features(train_df)
test = extract_features(test_df)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

print('Preparing features...')

num_feats = []
for col in train.columns:
    if any(s in col for s in ['len','wc','unique','avglen','isna','has_','salary_','loc_','telecommuting','has_company_logo','has_questions','unique_ratio','upper_ratio','digit_ratio','cp_','desc_']):
        if col not in ['fraudulent','id','title','company_profile','description','requirements','benefits','location','department','salary_range','employment_type','required_experience','required_education','industry','function','country']:
            num_feats.append(col)
num_feats = [c for c in num_feats if c in train.columns]

cat_feats = ['employment_type','required_experience','required_education','industry','function','department','country']

lbls = {}
for col in cat_feats:
    le = LabelEncoder()
    all_vals = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(all_vals)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    lbls[col] = le

X_num_train = train[num_feats + cat_feats].fillna(-1).astype(float).values
X_num_test = test[num_feats + cat_feats].fillna(-1).astype(float).values

In [ ]:
print('TF-IDF - Focus on company_profile and description!')

# company_profile TF-IDF
tfidf_cp = TfidfVectorizer(stop_words='english', max_features=600, ngram_range=(1,3), min_df=2, sublinear_tf=True)
tfidf_cp.fit(train['company_profile'].fillna(''))
X_cp_train = tfidf_cp.transform(train['company_profile'].fillna(''))
X_cp_test = tfidf_cp.transform(test['company_profile'].fillna(''))

# description TF-IDF
tfidf_desc = TfidfVectorizer(stop_words='english', max_features=800, ngram_range=(1,2), min_df=2, sublinear_tf=True)
tfidf_desc.fit(train['description'].fillna(''))
X_desc_train = tfidf_desc.transform(train['description'].fillna(''))
X_desc_test = tfidf_desc.transform(test['description'].fillna(''))

# requirements TF-IDF
tfidf_req = TfidfVectorizer(stop_words='english', max_features=400, ngram_range=(1,2), min_df=2, sublinear_tf=True)
tfidf_req.fit(train['requirements'].fillna(''))
X_req_train = tfidf_req.transform(train['requirements'].fillna(''))
X_req_test = tfidf_req.transform(test['requirements'].fillna(''))

# title TF-IDF
tfidf_title = TfidfVectorizer(stop_words='english', max_features=300, ngram_range=(1,2), min_df=2, sublinear_tf=True)
tfidf_title.fit(train['title'].fillna(''))
X_title_train = tfidf_title.transform(train['title'].fillna(''))
X_title_test = tfidf_title.transform(test['title'].fillna(''))

In [ ]:
X_train = sparse.hstack([
    sparse.csr_matrix(X_num_train),
    X_cp_train,
    X_desc_train,
    X_req_train,
    X_title_train
]).tocsr()

X_test = sparse.hstack([
    sparse.csr_matrix(X_num_test),
    X_cp_test,
    X_desc_test,
    X_req_test,
    X_title_test
]).tocsr()

y_train = train['fraudulent']
print(f'Final features: {X_train.shape}')

In [ ]:
print('\nTraining models...')
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
import xgboost as xgb

# XGBoost
model_xgb = xgb.XGBClassifier(
    n_estimators=2500,
    max_depth=11,
    learning_rate=0.007,
    subsample=0.90,
    colsample_bytree=0.65,
    gamma=0.1,
    reg_alpha=0.4,
    reg_lambda=2.8,
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict_proba(X_test)[:, 1]
print('XGBoost done')

In [ ]:
# Random Forest
model_rf = RandomForestClassifier(
    n_estimators=2500,
    max_depth=45,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict_proba(X_test)[:, 1]
print('RF done')

In [ ]:
# Extra Trees
model_et = ExtraTreesClassifier(
    n_estimators=2500,
    max_depth=45,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model_et.fit(X_train, y_train)
pred_et = model_et.predict_proba(X_test)[:, 1]
print('ET done')

In [ ]:
# HistGradientBoosting
model_hgb = HistGradientBoostingClassifier(
    max_iter=2000,
    learning_rate=0.015,
    max_depth=10,
    random_state=RANDOM_SEED
)
model_hgb.fit(X_train.toarray(), y_train)
pred_hgb = model_hgb.predict_proba(X_test.toarray())[:, 1]
print('HGB done')

In [ ]:
# Ensemble - XGBoost weighted highest
final_pred = (
    0.52 * pred_xgb + 
    0.20 * pred_rf + 
    0.15 * pred_et + 
    0.13 * pred_hgb
)

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'fraudulent': final_pred
})
submission.to_csv('submission.csv', index=False)

print('\n' + '='*80)
print('FOR NATIONAL HONOR - COMPLETE!')
print('='*80)
print(f'Predictions stats:')
print(f'  min: {final_pred.min():.6f}')
print(f'  max: {final_pred.max():.6f}')
print(f'  mean: {final_pred.mean():.6f}')
print(f'  >0.5: {(final_pred>0.5).sum()}')
print(f'  >0.9: {(final_pred>0.9).sum()}')
print(f'  >0.95: {(final_pred>0.95).sum()}')
print('submission.csv saved!')